# Fruit Detection with a Multi-Scale ResNet Detector

## Project Overview

This notebook trains a single-stage fruit detector for RGB images containing apples, bananas, oranges, and pears. The model predicts a class label, a normalized bounding box, and an objectness score for each detected fruit.

The original dataset is not distributed with this project. Provide a dataset following the annotation format below to reproduce the experiment.

## Model Goal

The goal is to localize and classify fruit instances in images. The detector uses a ResNet-18 backbone with two prediction heads: one on deeper features for larger objects and one on shallower features for smaller objects.

## Dataset Format

Expected structure:

```text
fruit-detection-challenge/
  detection/
    annotations/
      train.json
      val.json
      val.txt
      test.json        # optional for inference
    train/
    val/
  test/                # optional test images
```

The JSON annotations follow a COCO-style format with `images`, `annotations`, and `categories`. Bounding boxes are read as COCO `[x, y, width, height]` pixel coordinates and converted to normalized `[center_x, center_y, width, height]` boxes.

## How to Run

1. Install the dependencies listed below.
2. Place a compatible dataset in `fruit-detection-challenge/` or update the path constants in the setup cell.
3. Run the notebook from top to bottom to train and validate the detector.
4. The best validation checkpoint is reloaded for inference. An optional `test.json` enables test prediction export.

## Dependencies

Dependencies are specified in `pyproject.toml` and locked in `uv.lock`. Evaluation uses TorchMetrics with the pycocotools backend.

```bash
uv sync --locked
uv run jupyter lab
```

Select the project's `.venv` kernel. For a complete run with saved metrics and figures, use `uv run python scripts/run_notebook.py --output runs/my-run` from the repository root. See [README.md](README.md) for the annotation format and environment setup.


## Dataset Preparation


In [ ]:
%matplotlib inline

import json
import os
import random
from time import perf_counter
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms.functional as TF
from PIL import Image
from torch import nn
from torchvision.ops import distance_box_iou_loss, nms
from tqdm import tqdm

from object_detection_utils import (
    FileBasedAPCalculator,
    HungarianMatcher,
    get_src_permutation_idx,
)

DATASET_DIR = Path("fruit-detection-challenge")
DETECTION_DIR = DATASET_DIR / "detection"
ANNOTATION_DIR = DETECTION_DIR / "annotations"
TRAIN_JSON = ANNOTATION_DIR / "train.json"
VAL_JSON = ANNOTATION_DIR / "val.json"
VAL_GT_TXT = ANNOTATION_DIR / "val.txt"
TEST_JSON = ANNOTATION_DIR / "test.json"
TRAIN_IMAGE_DIR = DETECTION_DIR / "train"
VAL_IMAGE_DIR = DETECTION_DIR / "val"
TEST_IMAGE_DIR = DATASET_DIR / "test"

CLASS_NAMES = ["apple", "banana", "orange", "pear"]
IMAGE_SIZE = (512, 512)
SEED = 42
NUM_EPOCHS = 30
RUN_DIR = Path(os.environ.get("FRUIT_RUN_DIR", "runs/latest"))
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = RUN_DIR / "best_fruit_detector.pth"
PREDICTION_OUTPUT_PATH = RUN_DIR / "test_predictions.txt"
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.hub.set_dir(str(Path(".cache") / "torch"))

if not TRAIN_JSON.exists():
    print("Dataset not found. Add a compatible dataset under fruit-detection-challenge/ before training.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
class DetectionTrainTransform:
    def __init__(
        self,
        img_size=(256, 256),
        hflip_prob=0.5,
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05,
    ):
        self.img_size = img_size
        self.hflip_prob = hflip_prob
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation
        self.hue = hue

    def __call__(self, img, target):
        target_h, target_w = self.img_size

        scale = random.uniform(0.8, 1.2)
        new_h = int(target_h * scale)
        new_w = int(target_w * scale)
        img = TF.resize(img, (new_h, new_w))

        if scale >= 1.0:
            crop_top = (new_h - target_h) // 2
            crop_left = (new_w - target_w) // 2
            img = TF.crop(img, crop_top, crop_left, target_h, target_w)

            if target["boxes"].numel() > 0:
                boxes = target["boxes"].clone()
                px_cx = boxes[:, 0] * new_w - crop_left
                px_cy = boxes[:, 1] * new_h - crop_top
                px_w = boxes[:, 2] * new_w
                px_h = boxes[:, 3] * new_h

                x1 = (px_cx - px_w / 2).clamp(0, target_w)
                y1 = (px_cy - px_h / 2).clamp(0, target_h)
                x2 = (px_cx + px_w / 2).clamp(0, target_w)
                y2 = (px_cy + px_h / 2).clamp(0, target_h)

                valid = (x2 - x1 > 2) & (y2 - y1 > 2)
                new_cx = ((x1 + x2) / 2) / target_w
                new_cy = ((y1 + y2) / 2) / target_h
                new_w_box = (x2 - x1) / target_w
                new_h_box = (y2 - y1) / target_h

                target["boxes"] = torch.stack(
                    [new_cx, new_cy, new_w_box, new_h_box], dim=1
                )[valid]
                target["labels"] = target["labels"][valid]
        else:
            pad_top = (target_h - new_h) // 2
            pad_left = (target_w - new_w) // 2
            pad_bottom = target_h - new_h - pad_top
            pad_right = target_w - new_w - pad_left
            img = TF.pad(img, [pad_left, pad_top, pad_right, pad_bottom])

            if target["boxes"].numel() > 0:
                boxes = target["boxes"].clone()
                cx = (boxes[:, 0] * new_w + pad_left) / target_w
                cy = (boxes[:, 1] * new_h + pad_top) / target_h
                w = boxes[:, 2] * new_w / target_w
                h = boxes[:, 3] * new_h / target_h
                target["boxes"] = torch.stack([cx, cy, w, h], dim=1)

        if random.random() < self.hflip_prob:
            img = TF.hflip(img)
            if target["boxes"].numel() > 0:
                boxes = target["boxes"].clone()
                boxes[:, 0] = 1.0 - boxes[:, 0]
                target["boxes"] = boxes

        img = TF.adjust_brightness(img, 1.0 + random.uniform(-self.brightness, self.brightness))
        img = TF.adjust_contrast(img, 1.0 + random.uniform(-self.contrast, self.contrast))
        img = TF.adjust_saturation(img, 1.0 + random.uniform(-self.saturation, self.saturation))
        img = TF.adjust_hue(img, random.uniform(-self.hue, self.hue))

        return TF.to_tensor(img), target


class DetectionEvalTransform:
    def __init__(self, img_size=(256, 256)):
        self.img_size = img_size

    def __call__(self, img, target):
        img = TF.resize(img, self.img_size)
        return TF.to_tensor(img), target


class FruitDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, json_path, img_folder, img_size, transforms=None):
        self.img_folder = Path(img_folder)
        self.img_size = img_size
        self.transforms = transforms

        with open(json_path, "r", encoding="utf-8") as f:
            coco_data = json.load(f)

        self.images = coco_data["images"]
        self.category_id_to_index = {
            category["id"]: index for index, category in enumerate(coco_data["categories"])
        }
        self.img_id_to_anns = {}
        for annotation in coco_data["annotations"]:
            self.img_id_to_anns.setdefault(annotation["image_id"], []).append(annotation)

        self.img_files = [str(self.img_folder / image["file_name"]) for image in self.images]

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = self.img_files[idx]
        img = Image.open(img_path).convert("RGB")
        img_w, img_h = img.size

        boxes = []
        labels = []
        for annotation in self.img_id_to_anns.get(img_info["id"], []):
            x, y, w, h = annotation["bbox"]
            boxes.append([(x + w / 2) / img_w, (y + h / 2) / img_h, w / img_w, h / img_h])
            labels.append(self.category_id_to_index[annotation["category_id"]])

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32)
            if boxes
            else torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
        }

        if self.transforms:
            img, target = self.transforms(img, target)
        else:
            img = TF.resize(img, self.img_size)
            img = TF.to_tensor(img)

        return img, target

    def __len__(self):
        return len(self.images)


In [ ]:
train_transforms = DetectionTrainTransform(
    img_size=IMAGE_SIZE,
    hflip_prob=0.5,
    brightness=0.2,
    contrast=0.2,
    saturation=0.2,
    hue=0.05,
)
val_transforms = DetectionEvalTransform(img_size=IMAGE_SIZE)

train_dataset = FruitDetectionDataset(
    json_path=TRAIN_JSON,
    img_folder=TRAIN_IMAGE_DIR,
    img_size=IMAGE_SIZE,
    transforms=train_transforms,
)

val_dataset = FruitDetectionDataset(
    json_path=VAL_JSON,
    img_folder=VAL_IMAGE_DIR,
    img_size=IMAGE_SIZE,
    transforms=val_transforms,
)


def collate_fn(batch):
    return tuple(zip(*batch))


train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

print(f"Datasets ready: {len(train_dataset)} training images, {len(val_dataset)} validation images")


## Model Architecture


In [ ]:
class PredictionHead(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        hidden_channels = in_channels // 4

        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(in_channels // 2),
            nn.LeakyReLU(negative_slope=0.1),
            nn.Conv2d(in_channels // 2, hidden_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(negative_slope=0.1),
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(negative_slope=0.1),
        )

        self.box_predictor = nn.Sequential(
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(negative_slope=0.1),
            nn.Conv2d(hidden_channels, 4, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid(),
        )

        self.classifier = nn.Sequential(
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(negative_slope=0.1),
            nn.Conv2d(hidden_channels, num_classes, kernel_size=3, stride=1, padding=1),
        )

        self.objectness_predictor = nn.Sequential(
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(negative_slope=0.1),
            nn.Conv2d(hidden_channels, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        cls_logits = self.classifier(x)
        bbox_pred = self.box_predictor(x)
        objectness_pred = self.objectness_predictor(x)

        grid_h, grid_w = x.shape[-2], x.shape[-1]
        grid_y, grid_x = torch.meshgrid(
            torch.arange(grid_h, device=x.device),
            torch.arange(grid_w, device=x.device),
            indexing="ij",
        )

        cx = (bbox_pred[:, 0, :, :] + grid_x) / grid_w
        cy = (bbox_pred[:, 1, :, :] + grid_y) / grid_h
        w = bbox_pred[:, 2, :, :] * 2
        h = bbox_pred[:, 3, :, :] * 2
        bbox_pred = torch.stack([cx, cy, w, h], dim=1)

        return cls_logits, bbox_pred, objectness_pred


class ObjectDetectorMultiScale(nn.Module):
    def __init__(self, n_classes, pretrained=True):
        super().__init__()
        self.n_classes = n_classes

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-3])
        self.block_1 = nn.Sequential(*list(backbone.children())[-3])

        self.prediction_head = PredictionHead(512, self.n_classes)
        self.prediction_head_2 = PredictionHead(256, self.n_classes)

    def forward(self, x):
        shallow_features = self.backbone(x)
        deep_features = self.block_1(shallow_features)

        cls_logits, bbox_pred, objectness_pred = self.prediction_head(deep_features)
        cls_preds = cls_logits.flatten(2, 3).permute(0, 2, 1)
        box_preds = bbox_pred.flatten(2, 3).permute(0, 2, 1)
        obj_preds = objectness_pred.flatten(2, 3).permute(0, 2, 1)

        cls_logits_2, bbox_pred_2, objectness_pred_2 = self.prediction_head_2(shallow_features)
        cls_preds_2 = cls_logits_2.flatten(2, 3).permute(0, 2, 1)
        box_preds_2 = bbox_pred_2.flatten(2, 3).permute(0, 2, 1)
        obj_preds_2 = objectness_pred_2.flatten(2, 3).permute(0, 2, 1)

        return {
            "pred_logits": torch.cat([cls_preds, cls_preds_2], dim=1),
            "pred_boxes": torch.cat([box_preds, box_preds_2], dim=1),
            "pred_objectness": torch.cat([obj_preds, obj_preds_2], dim=1),
        }


model = ObjectDetectorMultiScale(n_classes=len(CLASS_NAMES), pretrained=True).to(device)


## Training Pipeline


In [ ]:
def cxcywh_to_xyxy_tensor(boxes):
    boxes = boxes.float()
    cx = boxes[:, 0].clamp(0.0, 1.0)
    cy = boxes[:, 1].clamp(0.0, 1.0)
    w = boxes[:, 2].clamp(min=1e-6, max=1.0)
    h = boxes[:, 3].clamp(min=1e-6, max=1.0)

    x1 = (cx - w / 2).clamp(0.0, 1.0)
    y1 = (cy - h / 2).clamp(0.0, 1.0)
    x2 = (cx + w / 2).clamp(0.0, 1.0)
    y2 = (cy + h / 2).clamp(0.0, 1.0)

    return torch.stack([x1, y1, x2, y2], dim=1)


def apply_classwise_nms(boxes, labels, scores, iou_thresh=0.70):
    if boxes.numel() == 0:
        return torch.empty((0,), dtype=torch.long)

    keep_indices = []
    for cls in labels.unique():
        cls_mask = labels == cls
        cls_boxes_xyxy = cxcywh_to_xyxy_tensor(boxes[cls_mask])
        keep = nms(cls_boxes_xyxy, scores[cls_mask], iou_thresh)
        keep_indices.append(torch.where(cls_mask)[0][keep])

    if not keep_indices:
        return torch.empty((0,), dtype=torch.long)

    return torch.cat(keep_indices)


In [ ]:
loss_history = []
map_history = []
map50_history = []
map75_history = []

val_predictions_path = RUN_DIR / "val_epoch_predictions.txt"
epoch_history = []
BEST_SCORE_THRESH = 0.05
BEST_NMS_THRESH = 0.70

optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": 1e-5},
    {"params": model.block_1.parameters(), "lr": 1e-5},
    {"params": model.prediction_head.parameters(), "lr": 1e-4},
    {"params": model.prediction_head_2.parameters(), "lr": 1e-4},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=10
)

num_epochs = NUM_EPOCHS
matcher = HungarianMatcher()
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_map = float("-inf")
best_epoch = 0
training_started = perf_counter()

for epoch in range(num_epochs):
    epoch_started = perf_counter()
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", mininterval=30)

    for images, targets in pbar:
        images = torch.stack([img.to(device, non_blocking=True) for img in images])
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            outputs = model(images)
            indices = matcher(outputs, targets)
            idx = get_src_permutation_idx(indices)

            num_boxes = max(sum(len(t["labels"]) for t in targets), 1)
            src_logits = outputs["pred_logits"]
            src_boxes = outputs["pred_boxes"]
            src_objectness = outputs["pred_objectness"]

            target_classes_o = torch.cat(
                [t["labels"][J] for t, (_, J) in zip(targets, indices)]
            )
            target_classes = torch.full(
                src_logits.shape[:2],
                -100,
                dtype=torch.int64,
                device=device,
            )
            target_classes[idx] = target_classes_o

            loss_ce = F.cross_entropy(
                src_logits.transpose(1, 2),
                target_classes,
                ignore_index=-100,
            ) if target_classes_o.numel() else src_logits.sum() * 0.0

            src_boxes_matched = src_boxes[idx]
            target_boxes = torch.cat(
                [t["boxes"][i] for t, (_, i) in zip(targets, indices)], dim=0
            )
            loss_bbox = F.l1_loss(
                src_boxes_matched,
                target_boxes,
                reduction="none",
            ).sum() / num_boxes

            loss_diou = distance_box_iou_loss(
                cxcywh_to_xyxy_tensor(src_boxes_matched),
                cxcywh_to_xyxy_tensor(target_boxes),
                reduction="mean",
            ) if src_boxes_matched.numel() else src_boxes_matched.sum() * 0.0

            batch_idx, matched_pred_indices = idx
            target_objectness = torch.zeros(
                src_objectness.shape[0],
                src_objectness.shape[1],
                device=device,
            )
            target_objectness[batch_idx, matched_pred_indices] = 1.0

        loss_objectness = F.binary_cross_entropy(
            src_objectness.squeeze(-1).float(),
            target_objectness.float(),
            reduction="mean",
        )

        losses = loss_ce + 5.0 * loss_bbox + 5.0 * loss_diou + 2.0 * loss_objectness
        if not torch.isfinite(losses):
            raise RuntimeError("Non-finite training loss; stop before saving invalid weights.")

        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += losses.item()
        pbar.set_postfix({
            "loss": f"{losses.item():.4f}",
            "ce": f"{loss_ce.item():.4f}",
            "bbox": f"{loss_bbox.item():.4f}",
            "diou": f"{loss_diou.item():.4f}",
            "obj": f"{loss_objectness.item():.4f}",
        }, refresh=False)

    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)

    model.eval()
    with torch.no_grad():
        with open(val_predictions_path, "w", encoding="utf-8") as f:
            for i in range(len(val_dataset)):
                img, _ = val_dataset[i]
                file_name = os.path.basename(val_dataset.img_files[i])
                outputs = model(img.unsqueeze(0).to(device))

                pred_boxes = outputs["pred_boxes"][0].detach().cpu()
                pred_logits = outputs["pred_logits"][0].detach().cpu()
                pred_objectness = outputs["pred_objectness"][0].detach().cpu()

                class_probs = F.softmax(pred_logits, dim=-1)
                final_scores = pred_objectness * class_probs
                max_scores, labels = torch.max(final_scores, dim=-1)

                keep = max_scores > BEST_SCORE_THRESH
                pred_boxes = pred_boxes[keep]
                labels = labels[keep] + 1
                max_scores = max_scores[keep]

                if len(pred_boxes) > 0:
                    keep_nms = apply_classwise_nms(
                        pred_boxes,
                        labels,
                        max_scores,
                        iou_thresh=BEST_NMS_THRESH,
                    )
                    pred_boxes = pred_boxes[keep_nms]
                    labels = labels[keep_nms]
                    max_scores = max_scores[keep_nms]

                for box, label, score in zip(pred_boxes, labels, max_scores):
                    cx, cy, w, h = box.tolist()
                    f.write(
                        f"{file_name}, {cx}, {cy}, {w}, {h}, {label.item()}, {score.item()}\n"
                    )

    ap_calculator = FileBasedAPCalculator(str(VAL_GT_TXT), val_predictions_path)
    ap = ap_calculator.calculate_map()

    mAP = ap["map"].item()
    map50 = ap["map_50"].item()
    map75 = ap["map_75"].item()

    map_history.append(mAP)
    map50_history.append(map50)
    map75_history.append(map75)
    scheduler.step(mAP)

    print(
        f"Epoch {epoch + 1} | "
        f"Avg Loss: {avg_loss:.4f} | "
        f"mAP: {mAP:.4f} | "
        f"mAP50: {map50:.4f} | "
        f"mAP75: {map75:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.2e}"
    )

    if mAP > best_map:
        best_map = mAP
        best_epoch = epoch + 1
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"New best mAP: {best_map:.4f} at epoch {best_epoch}. Model saved to {CHECKPOINT_PATH}.")

    epoch_history.append({
        "epoch": epoch + 1, "loss": avg_loss, "map": mAP,
        "map_50": map50, "map_75": map75,
        "seconds": perf_counter() - epoch_started,
        "backbone_lr": optimizer.param_groups[0]["lr"],
        "head_lr": optimizer.param_groups[2]["lr"],
    })
    (RUN_DIR / "history.json").write_text(json.dumps(epoch_history, indent=2), encoding="utf-8")

training_seconds = perf_counter() - training_started
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True))
model.eval()


## Evaluation


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training Loss", color="tab:blue")
ax1.plot(range(1, len(loss_history) + 1), loss_history, color="tab:blue", marker="o", label="Loss")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.set_ylabel("Validation mAP", color="tab:red")
ax2.plot(range(1, len(map_history) + 1), map_history, color="tab:red", marker="x", label="mAP")
ax2.plot(range(1, len(map50_history) + 1), map50_history, color="tab:green", marker=".", label="mAP50")
ax2.plot(range(1, len(map75_history) + 1), map75_history, color="tab:orange", marker=".", label="mAP75")
ax2.tick_params(axis="y", labelcolor="tab:red")

fig.legend(loc="upper center", ncol=4)
plt.title("Training Loss and Validation mAP")
fig.tight_layout()
plt.show()

print(f"Best epoch: {best_epoch}")
print(f"Best mAP: {best_map:.4f}")


## Inference / Prediction


In [ ]:
def save_predictions_to_txt(
    model,
    dataset,
    output_path,
    score_thresh=0.05,
    nms_thresh=0.70,
):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    model.eval()

    with torch.no_grad():
        with open(output_path, "w", encoding="utf-8") as f:
            for i in range(len(dataset.images)):
                img, _ = dataset[i]
                file_name = os.path.basename(dataset.img_files[i])
                outputs = model(img.unsqueeze(0).to(device))

                pred_boxes = outputs["pred_boxes"][0].detach().cpu()
                pred_logits = outputs["pred_logits"][0].detach().cpu()
                pred_objectness = outputs["pred_objectness"][0].detach().cpu()

                class_probs = F.softmax(pred_logits, dim=-1)
                final_scores = pred_objectness * class_probs
                max_scores, labels = torch.max(final_scores, dim=-1)

                keep = max_scores > score_thresh
                pred_boxes = pred_boxes[keep]
                labels = labels[keep]
                max_scores = max_scores[keep]

                if len(pred_boxes) > 0:
                    keep_nms = apply_classwise_nms(
                        pred_boxes,
                        labels,
                        max_scores,
                        iou_thresh=nms_thresh,
                    )
                    pred_boxes = pred_boxes[keep_nms]
                    labels = labels[keep_nms]
                    max_scores = max_scores[keep_nms]

                for box, label, score in zip(pred_boxes, labels, max_scores):
                    cx, cy, w, h = box.tolist()
                    f.write(
                        f"{file_name},{cx:.6f},{cy:.6f},{w:.6f},{h:.6f},{label.item() + 1},{score.item():.6f}\n"
                    )

    print(f"Saved predictions to: {output_path}")


In [ ]:
if TEST_JSON.exists() and TEST_IMAGE_DIR.exists():
    test_dataset = FruitDetectionDataset(
        json_path=TEST_JSON,
        img_folder=TEST_IMAGE_DIR,
        img_size=IMAGE_SIZE,
        transforms=DetectionEvalTransform(img_size=IMAGE_SIZE),
    )

    print(f"Test images: {len(test_dataset)}")
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True))
    model.to(device)

    save_predictions_to_txt(
        model,
        test_dataset,
        output_path=PREDICTION_OUTPUT_PATH,
        score_thresh=BEST_SCORE_THRESH,
        nms_thresh=BEST_NMS_THRESH,
    )
else:
    print("No test annotation file was found. Provide TEST_JSON and TEST_IMAGE_DIR before running inference.")


## Results and Discussion

The recorded seed-42 CPU run trained for 30 epochs on 218 images. The selected checkpoint was epoch 30, with **mAP50-95 = 0.4012**, **mAP50 = 0.8255**, and **mAP75 = 0.3206** on 10 validation images containing 72 objects. The checkpoint was reloaded and its score verified.

These are validation model-selection scores, not independent test results. Apples performed best; banana detections, overlapping fruit, duplicate boxes, and precise localization remain weaknesses. No comparison with another detector or single-head ablation was performed.

![Recorded training curves](results/seed42-cpu/training_curves.png)

The [complete report](results/seed42-cpu/README.md) includes per-class AP, all 10 validation photographs, exact settings, and machine-readable history. Its figures use a fixed display confidence of 0.30; metric evaluation uses confidence above 0.05.

The interactive visualization below compares ground-truth boxes in green against predictions in red. It is separate from the complete validation gallery.


In [ ]:
def box_iou_one(box_a, box_b):
    a = cxcywh_to_xyxy_tensor(box_a.unsqueeze(0))[0]
    b = cxcywh_to_xyxy_tensor(box_b.unsqueeze(0))[0]

    inter_w = torch.clamp(torch.min(a[2], b[2]) - torch.max(a[0], b[0]), min=0)
    inter_h = torch.clamp(torch.min(a[3], b[3]) - torch.max(a[1], b[1]), min=0)
    intersection = inter_w * inter_h
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - intersection

    return float((intersection / union).item()) if union > 0 else 0.0


def plot_detection_on_ax(idx, ax):
    img_tensor, target = val_dataset[idx]
    img_path = val_dataset.img_files[idx]
    pil_img = Image.open(img_path).convert("RGB")
    width, height = pil_img.size

    with torch.no_grad():
        outputs = model(img_tensor.unsqueeze(0).to(device))

    pred_boxes = outputs["pred_boxes"][0].detach().cpu()
    pred_logits = outputs["pred_logits"][0].detach().cpu()
    pred_objectness = outputs["pred_objectness"][0].detach().cpu()

    class_probs = F.softmax(pred_logits, dim=-1)
    final_scores = pred_objectness * class_probs
    scores, labels = torch.max(final_scores, dim=-1)

    keep = scores > BEST_SCORE_THRESH
    pred_boxes = pred_boxes[keep]
    labels = labels[keep]
    scores = scores[keep]

    if len(pred_boxes) > 0:
        keep_nms = apply_classwise_nms(pred_boxes, labels, scores, iou_thresh=BEST_NMS_THRESH)
        pred_boxes = pred_boxes[keep_nms]
        labels = labels[keep_nms]
        scores = scores[keep_nms]

    ax.imshow(pil_img)
    ax.set_title(f"Image {idx}: {os.path.basename(img_path)}", fontsize=10)

    for gt_box in target["boxes"]:
        x1, y1, x2, y2 = cxcywh_to_xyxy_tensor(gt_box.unsqueeze(0))[0].tolist()
        ax.add_patch(
            patches.Rectangle(
                (x1 * width, y1 * height),
                (x2 - x1) * width,
                (y2 - y1) * height,
                linewidth=1.5,
                edgecolor="lime",
                facecolor="none",
            )
        )

    for pred_box, pred_label, score in zip(pred_boxes, labels, scores):
        x1, y1, x2, y2 = cxcywh_to_xyxy_tensor(pred_box.unsqueeze(0))[0].tolist()
        same_class_ious = [
            box_iou_one(pred_box, gt_box)
            for gt_box, gt_label in zip(target["boxes"], target["labels"])
            if int(gt_label) == int(pred_label)
        ]
        best_iou = max(same_class_ious) if same_class_ious else 0.0

        ax.add_patch(
            patches.Rectangle(
                (x1 * width, y1 * height),
                (x2 - x1) * width,
                (y2 - y1) * height,
                linewidth=1.5,
                edgecolor="red",
                facecolor="none",
                linestyle="--",
            )
        )
        ax.text(
            x1 * width,
            max(0, y1 * height - 5),
            f"{CLASS_NAMES[int(pred_label)]} {score:.2f} IoU:{best_iou:.2f}",
            color="white",
            fontsize=7,
            bbox=dict(facecolor="red", alpha=0.6, edgecolor="none", pad=1),
        )

    ax.axis("off")


def show_validation_examples(start_idx=0):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    for offset, ax in enumerate(axes):
        current_idx = start_idx + offset
        if current_idx < len(val_dataset):
            plot_detection_on_ax(current_idx, ax)
        else:
            ax.axis("off")

    plt.tight_layout()
    plt.suptitle("Validation Results: Ground Truth (Green) vs Predictions (Red)", fontsize=16, y=1.02)
    plt.show()


show_validation_examples(start_idx=0)


## Resolution Preview

The following cell shows how resizing changes the input image and ground-truth boxes. It does not train or benchmark separate models at different resolutions.


In [ ]:
def draw_boxes_on_axis(ax, image, target, title):
    width, height = image.size
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")

    for box, label in zip(target["boxes"], target["labels"]):
        cx, cy, w, h = box.tolist()
        x1 = (cx - w / 2) * width
        y1 = (cy - h / 2) * height
        box_w = w * width
        box_h = h * height

        ax.add_patch(
            patches.Rectangle(
                (x1, y1),
                box_w,
                box_h,
                linewidth=2,
                edgecolor="lime",
                facecolor="none",
            )
        )
        ax.text(
            x1,
            max(0, y1 - 10),
            CLASS_NAMES[int(label)],
            color="black",
            fontsize=8,
            bbox=dict(facecolor="lime", alpha=0.7, edgecolor="none"),
        )


def show_resolution_comparison(indices=None):
    if indices is None:
        indices = list(range(min(3, len(val_dataset))))

    for idx in indices:
        _, target = val_dataset[idx]
        img_path = val_dataset.img_files[idx]
        original = Image.open(img_path).convert("RGB")

        img_256 = original.resize((256, 256))
        img_512 = original.resize((512, 512))

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        file_name = os.path.basename(img_path)
        draw_boxes_on_axis(axes[0], img_256, target, f"{file_name} | 256x256")
        draw_boxes_on_axis(axes[1], img_512, target, f"{file_name} | 512x512")
        plt.tight_layout()
        plt.show()


show_resolution_comparison()


## Future Improvements

- Collect a larger, scene-separated test set and report results across multiple random seeds.
- Compare established detector baselines under the same data split and training budget.
- Ablate the second prediction head to measure its contribution.
- Investigate banana localization and duplicate detections in crowded scenes.
- Tune score and NMS thresholds on validation data, then freeze them before independent testing.
